# **Fine-tune a pre-trained Large Language Model (LLM) on a specialized task using parameter-efficient fine-tuning methods like LoRA or Adapter Layers.**

## Install necessary libraries

### Subtask:
Install libraries required for fine-tuning LLMs, such as `transformers`, `peft`, and a deep learning framework like `PyTorch` or `TensorFlow`.


In [ ]:
%pip install transformers peft torch

## Load a pre-trained llm and tokenizer

### Subtask:
Choose a pre-trained LLM from a library like `transformers` and load it along with its corresponding tokenizer.


**Reasoning**:
Import the necessary classes from the `transformers` library and load a pre-trained model and its tokenizer.



In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"Model: {model_name} loaded successfully.")
print(f"Tokenizer: {model_name} loaded successfully.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model: gpt2 loaded successfully.
Tokenizer: gpt2 loaded successfully.


## Prepare the dataset

### Subtask:
Load and preprocess the dataset for your specialized task. This may involve tokenization, formatting, and splitting into training and validation sets.


**Reasoning**:
Load and preprocess the dataset, including tokenization, formatting, and splitting into training and validation sets.



In [ ]:
from datasets import load_dataset

dataset = load_dataset('nyu-mll/glue','mrpc')

print(dataset)


DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})


In [ ]:
# Add a padding token to the tokenizer
tokenizer.pad_token = tokenizer.eos_token

# Tokenize the dataset again
def tokenize_function(examples):
    return tokenizer(examples["sentence1"], examples["sentence2"], truncation=True, padding="max_length")

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Format the tokenized dataset
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

# Split the dataset
train_dataset = tokenized_datasets["train"]
validation_dataset = tokenized_datasets["validation"]

print("Dataset loaded, tokenized, formatted, and split successfully.")
print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(validation_dataset)}")

Dataset loaded, tokenized, formatted, and split successfully.
Training dataset size: 3668
Validation dataset size: 408


## Configure parameter-efficient fine-tuning

### Subtask:
Choose a parameter-efficient fine-tuning method (e.g., LoRA, Adapter Layers) and configure its parameters using the `peft` library.


**Reasoning**:
Import the LoRA configuration class from the peft library and create a LoRA configuration object with specified parameters.



In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,  # Rank of the update matrices
    lora_alpha=16,  # Scaling factor
    lora_dropout=0.1,  # Dropout rate
    bias="none", # Bias type
    task_type="CAUSAL_LM", # Task type
)

print(lora_config)

LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules=None, exclude_modules=None, lora_alpha=16, lora_dropout=0.1, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, ensure_weight_tying=False)


## Integrate peft with the llm

### Subtask:
Integrate the PEFT configuration with the loaded pre-trained LLM.


**Reasoning**:
Integrate the PEFT configuration with the loaded pre-trained LLM by using the `get_peft_model` function and print the resulting PEFT model.



In [ ]:
import torchao
!pip install --upgrade torchao
from peft import get_peft_model

peft_model = get_peft_model(model, lora_config)
print(peft_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GPT2LMHeadModel(
      (transformer): GPT2Model(
        (wte): Embedding(50257, 768)
        (wpe): Embedding(1024, 768)
        (drop): Dropout(p=0.1, inplace=False)
        (h): ModuleList(
          (0-11): 12 x GPT2Block(
            (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (attn): GPT2Attention(
              (c_attn): lora.Linear(
                (base_layer): Conv1D(nf=2304, nx=768)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=768, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2304, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
           

/usr/local/lib/python3.13/dist-packages/peft/tuners/lora/layer.py:2631: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


## Set up the training process

### Subtask:
Define the training arguments, optimizer, and loss function.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",  # Output directory
    num_train_epochs=3,  # Number of training epochs
    per_device_train_batch_size=4,  # Batch size per device during training
    per_device_eval_batch_size=4,   # Batch size for evaluation
    learning_rate=3e-4,  # Learning rate
    save_steps=10, # Save checkpoint every X updates steps
    logging_steps=10, # Log every X steps instead
    report_to=[], # Set to empty list to disable all reporting
)

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=False,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_star

## Train the model

### Subtask:
Train the model using the prepared dataset and the configured PEFT method.


In [ ]:
import torch
from transformers import Trainer

# Create the labels column for causal language modeling
def create_labels(examples):
    examples["labels"] = [ids.tolist() for ids in examples["input_ids"]]
    for i in range(len(examples["labels"])):
        for j in range(len(examples["labels"][i])):
            if examples["labels"][i][j] == tokenizer.pad_token_id:
                examples["labels"][i][j] = -100
    return examples

# Map labels and select a smaller subset to speed up training
train_dataset = train_dataset.map(create_labels, batched=True).select(range(500))
validation_dataset = validation_dataset.map(create_labels, batched=True).select(range(200))

# Re-instantiate the Trainer with the smaller datasets
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
)

# Start the training process
trainer.train()

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.564073
20,3.487613
30,3.184845
40,3.271705
50,3.046651
60,3.064749
70,3.142266
80,2.928776
90,3.083252
100,3.018827


TrainOutput(global_step=375, training_loss=2.940896779378255, metrics={'train_runtime': 385.5712, 'train_samples_per_second': 3.89, 'train_steps_per_second': 0.973, 'total_flos': 786594004992000.0, 'train_loss': 2.940896779378255, 'epoch': 3.0})

## Evaluate the model

### Subtask:
Evaluate the performance of the fine-tuned model on the validation set.


In [ ]:
evaluation_results = trainer.evaluate()
print(evaluation_results)

Training Loss,Validation Loss,Step
2.778358,2.694510,375


{'eval_loss': 2.694509983062744}


## Save the fine-tuned model

### Subtask:
Save the fine-tuned model for future use.


In [ ]:
output_dir = "./fine_tuned_model"
peft_model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Fine-tuned model and tokenizer saved to {output_dir}")

Fine-tuned model and tokenizer saved to ./fine_tuned_model


### Optimization: Increasing LoRA Rank and Tuning Hyperparameters
To improve the model's performance, we can increase the `r` parameter in the LoRA configuration and fine-tune the learning rate and batch size.

In [ ]:
from peft import get_peft_model, LoraConfig

# Optimized LoRA Config
lora_config_opt = LoraConfig(
    r=16,  # Increased rank
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Re-wrap the base model
opt_model = get_peft_model(model, lora_config_opt)

# Optimized Training Arguments
opt_training_args = TrainingArguments(
    output_dir="./optimized_results",
    num_train_epochs=5, # Slightly more epochs for the subset
    per_device_train_batch_size=8,
    learning_rate=1e-4, # Lower learning rate for stability
    logging_steps=5,
    report_to=[]
)

# Initialize Trainer
opt_trainer = Trainer(
    model=opt_model,
    args=opt_training_args,
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
)

opt_trainer.train()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Step,Training Loss
5,3.506310
10,3.572138
15,3.366253
20,3.453881
25,3.247461
30,3.333274
35,3.389013
40,3.190288
45,3.353895
50,3.263699


TrainOutput(global_step=315, training_loss=3.0364854419042193, metrics={'train_runtime': 625.3953, 'train_samples_per_second': 3.997, 'train_steps_per_second': 0.504, 'total_flos': 1315519856640000.0, 'train_loss': 3.0364854419042193, 'epoch': 5.0})

### Inference Check
Let's generate some text to see how the model performs after these optimizations.

In [ ]:
inputs = tokenizer("The quick brown fox", return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = opt_model.generate(input_ids=inputs["input_ids"], max_new_tokens=50)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

The quick brown fox was a favorite of the American public because it was a good looking fox and was a good looking fox. The quick brown fox was a favorite of the American public because it was a good looking fox and was a good looking fox. The quick brown fox


### Improved Inference
Using sampling and repetition penalty to fix the looping behavior seen above.

In [ ]:
# Generate text with improved sampling parameters
inputs = tokenizer("The quick brown fox", return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = opt_model.generate(
        input_ids=inputs["input_ids"],
        max_new_tokens=50,
        do_sample=True,
        top_p=0.92,
        top_k=50,
        repetition_penalty=1.2,
        temperature=0.7
    )

print("Optimized Generation:")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Optimized Generation:
The quick brown foxes of the world have been spotted in one part, they say , which has its own distinctive red eyes .
And that's exactly what happens when it comes to play : The birds are seen watching from a small group atelier on south-
